In [ ]:
from typing import List, Tuple
from dataclasses import dataclass

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score

from dataUtils import pages_with_annotations
from transformers import BertModel, BertConfig
from transformers import AutoModel, AutoTokenizer

ModuleNotFoundError: No module named 'dataUtils'

### Elman RNN inference

In [ ]:
# Different types of RNNs, biRNNs, and multi-layered RNNs 
class NERModelElman(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int,
                 embedding_model: AutoTokenizer, num_layers: int = 2, num_tags: int = 3):
        super().__init__()
        self.embedding_model = embedding_model
        self.classifier = nn.Linear(hidden_dim, num_tags)
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers)

    def forward(self, x: torch.tensor) -> torch.tensor:
        """x is tensor of shape BxS where B is batch size and S is sequence length of inputs"""
        with torch.no_grad():
            emb_x = self.embedding_model(x).last_hidden_state
        emb_x = torch.permute(emb_x, (1, 0, 2)) 
        out, _ = self.rnn(emb_x)
        out = torch.permute(out, (1, 0, 2))
        y = self.classifier(out) # B x S x E
        return y